In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr

import hvplot.xarray
import hvplot.pandas
import holoviews as hv
from bokeh.models import HoverTool

from clearwater_modules_v2.config import init_from_file
from clearwater_riverine.plotting import RiverinePlotter


In [ ]:
#### Set local filepath to the configuration yaml file ####
#  Set the following "use_v1_intrp" variable to "False" to explore the effects of the improved 
#  interpolation scheme for boundary condition input time series. Keeping it as "True" will
#  demostrates version 2 can produce the same results as version 1:
use_v1_intrp = True
#use_v1_intrp = False

if use_v1_intrp == True:
    config_path = Path(r"D:\Clearwater\ClearWater-modules\examples\data_temp\sumwere_creek_coarse_p48\modules_v1_interp.yml")
else:
    config_path = Path(r"D:\Clearwater\ClearWater-modules\examples\data_temp\sumwere_creek_coarse_p48\modules.yml")

print(config_path.exists())

#### Initialize a version 2 model of the linked Riverine and TSM models ####
model = init_from_file(config_path)

#### Simulate a version 2 model of the linked Riverine and TSM models ####
model.run()

In [ ]:
#Initialize a Riverine dynamic plotting tool
plotter = RiverinePlotter(registry=model._Model__registry, crs='EPSG:26916')

In [ ]:
#Plot the water temperature results from the linked Riverine and TSM model simulation
plotter.dynamic_plot(constituent_name = 'water_temperature')

In [ ]:
#### Open a Zarr directory for a previously saved version 1 reaction model output ####

reaction_model_v1_savePath = Path(r'D:\Clearwater\ClearWater-modules\examples\data_temp\sumwere_creek_coarse_p48\model_output_v1_asPrevPresented\model_output_reaction.zarr')
print(reaction_model_v1_savePath.exists())

reaction_model_v1 = xr.open_zarr(reaction_model_v1_savePath)
reaction_model_v1

In [ ]:
def compare_v1_v2_for_nface_i(
        nface_i: int,
        reaction_model_v1: xr.Dataset,
        model: clearwater_modules_v2.model.Model
):
    """
    Returns a dataframe for a nface with water temperature model results from a version 1 and version 2 simulations
    and plots
    """
    key_v1 = 'water_temp_c'
    key_v2 = 'water_temperature'
    
    #Format Select nface model results into a dataframe for version 1
    waterTemp_v1 = reaction_model_v1[key_v1].isel(nface=nface_i).to_dataframe()
    waterTemp_v1['old_time'] = waterTemp_v1.index
    waterTemp_v1['total_seconds'] = waterTemp_v1.index * 30  #convert from timestep index to total seconds (30 second timesteps)
    waterTemp_v1['time'] = waterTemp_v1['time'] + pd.to_timedelta(waterTemp_v1.total_seconds, unit='s')
    waterTemp_v1['temp_v1'] = waterTemp_v1['water_temp_c']
    waterTemp_v1_B = waterTemp_v1[['time', 'temp_v1']]
    waterTemp_v1_C = waterTemp_v1_B.iloc[:-1] # remove last record
    
    #Format Select nface model results into a dataframe for version 2
    waterTemp_v2 = model._Model__registry._registry[key_v2].get().isel(nface=nface_i).to_dataframe()
    waterTemp_v2_B = waterTemp_v2.rename(columns={'water_temperature': 'temp_v2'})
    waterTemp_v2_C = waterTemp_v2_B[['temp_v2']]
    
    #Merge dataframes
    df_merged = pd.merge(waterTemp_v1_C, waterTemp_v2_C, on='time', suffixes=('_v1', '_v2'))
    
    #Calculate difference between v1 and v2 water temperature
    df_merged['diff_v1-v2'] = df_merged['temp_v1'] - df_merged['temp_v2']


    #### PLOTTING ####
    hover = HoverTool(tooltips=[
        ("Date/Time", "@time{%F %T}"),
        ("v1", "@temp_v1{0.00000000}"),
        ("v2", "@temp_v2{0.00000000}"),
    ],
        formatters={
            "@time": "datetime",
        },
        mode='vline'
    )
    
    curve1 = hv.Curve(
        df_merged,
        kdims=['time'],
        vdims=['temp_v1', 'temp_v2'],
        label='v1'
    )
    
    curve2 = hv.Curve(
        df_merged,
        kdims=['time'],
        vdims=['temp_v2', 'temp_v1'],
        label='v2'
    ).opts(line_dash='dashed')
    
    plot_curves = (curve1 * curve2).opts(
        hv.opts.Overlay(
            width=700,
            height=500,
            xlabel='Date/Time',
            ylabel='Water Temperature',        
            legend_position='bottom',
            show_legend=True,
            legend_opts={'location': 'bottom_center', 'orientation': 'horizontal'}
        ),
        hv.opts.Curve(tools=[hover])
    )

    plot_diff = df_merged.hvplot(x='time', y='diff_v1-v2', label='diff_v1-v2', hover=True)
    plot_compare = (plot_curves + plot_diff).cols(1)
    
    return df_merged, plot_compare
    

In [ ]:
nface_i = 275 #### A Model Cell in the Upstream End of Sumwere Creek ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

In [ ]:
nface_i = 226 #### A Model Cell at the Confluence of Sumwere Creek and Cold Spring ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

In [ ]:
nface_i = 150 #### A Model Cell in the Oxbow Lake near the Power Plant discharge ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot

In [ ]:
nface_i = 264 #### A Model Cell in the Downstream End of Sumwere Creek ####

df, plot = compare_v1_v2_for_nface_i(nface_i, reaction_model_v1, model)
plot